### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


# Lecture 9: Regularization, Noise, Data Size, and Model Complexity

Controls included:
- noise level,
- number of data points,
- regularization strength $\lambda$,
- model complexity (polynomial degree),
- ground-truth noise.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

rng = np.random.default_rng(12)


def make_data(n=40, gt_noise=0.0, obs_noise=0.3):
    x = np.linspace(-3, 3, n)
    y_star = 1.0 + 0.7*x - 0.35*x**2 + 0.06*x**3 + rng.normal(0, gt_noise, n)
    y = y_star + rng.normal(0, obs_noise, n)
    return x, y, y_star


def design(x, deg):
    return np.vstack([x**d for d in range(deg+1)]).T


def ridge_fit(X, y, lam):
    I = np.eye(X.shape[1])
    I[0,0] = 0.0
    return np.linalg.pinv(X.T@X + lam*I) @ X.T @ y


def draw(n_points=40, noise=0.3, lambda_=0.5, degree=5, gt_noise=0.0):
    x, y, y_star = make_data(n=n_points, gt_noise=gt_noise, obs_noise=noise)
    X = design(x, degree)
    th = ridge_fit(X, y, lambda_)

    xg = np.linspace(x.min(), x.max(), 300)
    yg = design(xg, degree) @ th

    mse = np.mean((design(x, degree)@th - y)**2)
    coef_norm = np.linalg.norm(th[1:])

    fig, axs = plt.subplots(1, 2, figsize=(13, 4.8))
    axs[0].scatter(x, y, s=18, alpha=0.75, label='observed')
    axs[0].plot(x, y_star, color='gray', lw=1.5, alpha=0.8, label='GT (noisy)')
    axs[0].plot(xg, yg, 'r', lw=2.2, label='ridge fit')
    axs[0].set_title('Fit under noise / complexity / lambda')
    axs[0].legend(loc='best')

    axs[1].bar(np.arange(len(th)), th, color='#4b7bec')
    axs[1].set_title(f'Coefficients | ||w||={coef_norm:.3f} | train MSE={mse:.3f}')
    axs[1].set_xlabel('coefficient index')

    plt.tight_layout()
    plt.show()

widgets.interact(
    draw,
    n_points=widgets.IntSlider(description='n_points', min=20, max=200, step=5, value=40, continuous_update=False),
    noise=widgets.FloatSlider(description='noise', min=0.05, max=1.2, step=0.05, value=0.3, continuous_update=False),
    lambda_=widgets.FloatLogSlider(description='lambda', base=10, min=-3, max=2, step=0.1, value=0.5, continuous_update=False),
    degree=widgets.IntSlider(description='complexity', min=1, max=12, value=5, continuous_update=False),
    gt_noise=widgets.FloatSlider(description='gt_noise', min=0.0, max=0.8, step=0.05, value=0.0, continuous_update=False),
)
